# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

# stage.stg_pointhistory

**목적**: `accounts_pointhistory`를 분석용 Stage로 정제
**산출물**: `stage.stg_pointhistory` (VIEW)

**핵심 결정 요약**
- 중복 1,939 제거 & 결측은 추후 EDA시 판단
-

**Raw 품질**: 중복 1,939(제거) · 결측 2,992 · status 예상 외 값 0 · PK(id) 유니크

## Raw 품질 확인

### 데이터 샘플 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT *
FROM `your-gcp-project.raw.accounts_pointhistory`
LIMIT 5;

### 1. 컬럼 타입 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    column_name,
    data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'accounts_pointhistory';

#### KST 변환

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  id,
  user_id,
  user_question_record_id,
  delta_point,
  -- 한국 시간(KST)만 단독으로 추출
  DATETIME(created_at, 'Asia/Seoul') AS created_at
FROM `your-gcp-project.raw.accounts_pointhistory`
LIMIT 5;

### 2. 중복, 결측치, 이상치 확인

In [ ]:
%%bigquery --project your-gcp-project
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
)
SELECT
  COUNT(*) AS total_rows,
  -- 중복 행 수 확인 (전체 행 수 - 고유 행 수)
  COUNT(*) - COUNT(DISTINCT CONCAT(
    COALESCE(CAST(user_id AS STRING), 'NULL'), '_',
    COALESCE(CAST(user_question_record_id AS STRING), 'NULL'), '_',
    COALESCE(CAST(delta_point AS STRING), 'NULL'), '_',
    COALESCE(CAST(created_at AS STRING), 'NULL')
  )) AS duplicate_rows,
  -- 결측치 확인
  COUNTIF(user_id IS NULL) AS null_user_id,
  COUNTIF(user_question_record_id IS NULL) AS null_user_question_record_id,
  COUNTIF(delta_point IS NULL) AS null_delta_point,
  COUNTIF(created_at IS NULL) AS null_created_at,
  -- 이상치 확인 (예: 포인트 변동량이 0인 경우 등 필요에 따라 조건 조정 가능)
  COUNTIF(delta_point = 0) AS zero_delta_point_count
FROM parsed;

#### 중복 제거

In [ ]:
%%bigquery --project your-gcp-project
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
),
deduped AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    created_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id, user_question_record_id, delta_point, created_at
      ORDER BY created_at
    ) as row_num
  FROM parsed
)
SELECT
  user_id,
  user_question_record_id,
  delta_point,
  created_at
FROM deduped
WHERE row_num = 1;

In [ ]:
%%bigquery --project your-gcp-project
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
),
deduped AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    created_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id, user_question_record_id, delta_point, created_at
      ORDER BY created_at
    ) as row_num
  FROM parsed
)
SELECT
  COUNT(*) AS deduped_total_rows,  -- 중복 제거 후 총 행 수
  COUNT(*) - COUNT(DISTINCT CONCAT(
    COALESCE(CAST(user_id AS STRING), 'NULL'), '_',
    COALESCE(CAST(user_question_record_id AS STRING), 'NULL'), '_',
    COALESCE(CAST(delta_point AS STRING), 'NULL'), '_',
    COALESCE(CAST(created_at AS STRING), 'NULL')
  )) AS remaining_duplicate_rows    -- 남은 중복 수 (0이 나오면 정상)
FROM deduped
WHERE row_num = 1;

### 3.분포 및 최댓값/최소값 검증

In [ ]:
%%bigquery --project your-gcp-project
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
),
deduped AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    created_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id, user_question_record_id, delta_point, created_at
      ORDER BY created_at
    ) AS row_num
  FROM parsed
)
SELECT
  -- 1. 포인트 변동량(delta_point) 수치 이상치 점검
  MIN(delta_point) AS min_delta_point,             -- 최솟값 (음수가 너무 크지 않은지)
  MAX(delta_point) AS max_delta_point,             -- 최댓값 (비정상적으로 큰 포인트 지급이 있는지)
  AVG(delta_point) AS avg_delta_point,             -- 평균 변동량
  COUNTIF(delta_point < 0) AS negative_point_cnt,  -- 포인트 차감 건수
  COUNTIF(delta_point = 0) AS zero_point_cnt,      -- 0포인트 변동 건수

  -- 2. 생성 시간(created_at) 시간 이상치 점검
  MIN(created_at) AS min_created_at,               -- 가장 과거 기록 시간
  MAX(created_at) AS max_created_at,               -- 가장 최근 기록 시간
  COUNTIF(created_at > CURRENT_DATETIME('Asia/Seoul')) AS future_created_at_cnt -- 미래 시점 로그 존재 여부
FROM deduped
WHERE row_num = 1;

### 4. user_id / user_question_record_id 음수 ID 존재 여부

In [ ]:
%%bigquery --project your-gcp-project
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
),
deduped AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    created_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id, user_question_record_id, delta_point, created_at
      ORDER BY created_at
    ) AS row_num
  FROM parsed
)
SELECT
  COUNTIF(user_id <= 0) AS invalid_user_id_cnt,
  COUNTIF(user_question_record_id <= 0) AS invalid_record_id_cnt
FROM deduped
WHERE row_num = 1;

### 5. 다른 테이블(users)과의 외래키(FK) 정합성 확인

In [ ]:
%%bigquery --project your-gcp-project
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
),
deduped AS (
  SELECT DISTINCT user_id
  FROM parsed
)
SELECT
  COUNT(d.user_id) AS total_point_users,
  COUNT(u.id) AS matched_users,
  COUNT(d.user_id) - COUNT(u.id) AS unmapped_users -- 0이면 모든 포인트 기록 유저가 users 테이블에 존재
FROM deduped d
LEFT JOIN `your-gcp-project.raw.raw_user` u
  ON d.user_id = u.id;

## 최종 Stage

### view (관례)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_pointhistory` AS
WITH parsed AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    DATETIME(created_at, 'Asia/Seoul') AS created_at
  FROM `your-gcp-project.raw.accounts_pointhistory`
),
deduped AS (
  SELECT
    user_id,
    user_question_record_id,
    delta_point,
    created_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id, user_question_record_id, delta_point, created_at
      ORDER BY created_at
    ) AS row_num
  FROM parsed
)
SELECT
  user_id,
  user_question_record_id,
  delta_point,
  created_at
FROM deduped
WHERE row_num = 1;

### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT user_id) AS unique_users
FROM `your-gcp-project.stage.stg_pointhistory`;